# Fine-tuning Qwen2.5-Math with QLoRA: V1 and V4

I wanted to see whether fine-tuning an already capable mathematics model on GSM8K would make it better at solving word problems. I tried several settings, but this notebook focuses on the two most useful experiments: **V1**, my first working QLoRA run, and **V4**, where I trained more parts of the model with LoRA.

The interesting result was that **V4 had a lower validation loss but answered fewer test questions correctly**. This notebook shows the data preparation, training settings, and evaluation used to investigate that result.

**Model:** `Qwen/Qwen2.5-Math-1.5B-Instruct`  
**Dataset:** `openai/gsm8k` (`main`)  
**Hardware used:** RTX 3070 Laptop GPU, 8 GB VRAM  
**Method:** 4-bit QLoRA, supervised fine-tuning

> The results near the end are from my earlier runs. Cell outputs have been cleared for GitHub; rerunning training will take time and may produce slightly different numbers.

## 1. Set up the environment

I used PyTorch, Transformers, PEFT, bitsandbytes and Datasets. The code assumes a CUDA-enabled NVIDIA GPU. Install the packages in your own environment before starting.

```bash
pip install torch transformers peft accelerate bitsandbytes datasets pandas tqdm
```

A note on memory: don't keep V1 and V4 loaded on the GPU at the same time. Each experiment below loads a fresh copy of the base model.

In [1]:
import gc
import re

import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)

MODEL_ID = "Qwen/Qwen2.5-Math-1.5B-Instruct"
SEED = 42

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce RTX 3070 Laptop GPU


## 2. Prepare GSM8K

GSM8K contains grade-school mathematics word problems with worked solutions. I split its training portion into **90% training and 10% validation**, then kept the official test set separate.

The original answers contain calculator-style annotations such as `<<...>>` and a `####` marker before the final number. I removed the annotations and replaced the marker with `Final answer:` so the target format was easier to read.

In [2]:
dataset = load_dataset("openai/gsm8k", "main")
split = dataset["train"].train_test_split(test_size=0.1, seed=SEED)

def clean_math_example(example):
    answer = re.sub(r"<<.*?>>", "", example["answer"])
    return {"answer": answer.replace("####", "Final answer:")}

train_clean = split["train"].map(clean_math_example)
validation_clean = split["test"].map(clean_math_example)
test_clean = dataset["test"].map(clean_math_example)

print("Training:", len(train_clean))
print("Validation:", len(validation_clean))
print("Test:", len(test_clean))
print("\nExample question:", train_clean[0]["question"])
print("\nExample answer:", train_clean[0]["answer"])

Map:   0%|          | 0/6725 [00:00<?, ? examples/s]

Map:   0%|          | 0/748 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Training: 6725
Validation: 748
Test: 1319

Example question: A kilogram of pork costs $6 while a kilogram of chicken costs $2 less. How much will a 3-kilogram of chicken and a kilogram of pork cost?

Example answer: A kilogram of chicken costs $6 - $2 = $4.
Three kilograms of chicken cost $4 x 3 = $12.
So, a 3-kilogram of chicken and a kilogram of pork cost $12 + $6 = $18.
Final answer: 18


## 3. Load the model in 4-bit

The original model has about 1.5 billion parameters. Loading it in 4-bit reduces the GPU memory needed for its frozen weights. LoRA then adds small trainable matrices instead of updating the whole model.

I use the same loading function for V1 and V4 so that each experiment starts from the same base model.

In [3]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_fresh_model():
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
        dtype=torch.float16,
    )
    base_model.config.use_cache = False
    return prepare_model_for_kbit_training(base_model)

## 4. Tokenize questions and solutions

For each example, the model sees a system instruction and a question, followed by the worked answer. During training, I mask the prompt tokens with `-100` so the loss is calculated only on the assistant's solution.

The tokenizer's chat template makes sure the text is formatted the way Qwen expects. I also check that the prompt is really a prefix of the full conversation, so the labels line up correctly.

In [8]:
SYSTEM_INSTRUCTION = (
    "You are a mathematics tutor. "
    "Solve the problem step by step. "
    "End with 'Final answer: <number>'."
)

def token_ids(messages, add_generation_prompt=False):
    result = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=add_generation_prompt,
    )
    # Transformers versions can return either a list or a dictionary.
    return result["input_ids"] if isinstance(result, dict) else result




def tokenize_math_example(example):
    # Prompt without the assistant's solution
    prompt_messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": example["question"]}
    ]

    prompt_tokens = token_ids(prompt_messages, add_generation_prompt=True)

    # Full conversation including the solution
    full_messages = prompt_messages + [{"role": "assistant", "content": example["answer"]}]

    full_tokens = token_ids(full_messages, add_generation_prompt=False)

    prompt_ids = prompt_tokens["input_ids"]
    full_ids = full_tokens["input_ids"]

    # Ignore prompt tokens when calculating loss
    labels = ([-100] * len(prompt_ids) + full_ids[len(prompt_ids):])

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels
    }

train_tokenized = train_clean.map(
    tokenize_math_example, remove_columns=train_clean.column_names
)
validation_tokenized = validation_clean.map(
    tokenize_math_example, remove_columns=validation_clean.column_names
)

example = train_tokenized[0]
print("Sequence length:", len(example["input_ids"]))
print("Masked prompt tokens:", example["labels"].count(-100))
print("Solution tokens:", sum(x != -100 for x in example["labels"]))

Map:   0%|          | 0/6725 [00:00<?, ? examples/s]

Map:   0%|          | 0/748 [00:00<?, ? examples/s]

Sequence length: 139
Masked prompt tokens: 73
Solution tokens: 66


### Padding each batch

Questions and solutions have different lengths. Instead of padding every example to a fixed size, this collator pads only to the longest sequence in the current batch.

The padding token in `labels` is `-100`, which tells the loss function to ignore it.

In [9]:
def math_data_collator(features):

    max_len = max(len(f["input_ids"]) for f in features)

    batch = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []}

    for f in features:
        pad_len = max_len - len(f["input_ids"])
        batch["input_ids"].append(f["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        batch["attention_mask"].append(f["attention_mask"] + [0] * pad_len)

        batch["labels"].append(f["labels"] + [-100] * pad_len)

    return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}

batch = math_data_collator([train_tokenized[0], train_tokenized[1]])
print({name: tuple(value.shape) for name, value in batch.items()})

{'input_ids': (2, 180), 'attention_mask': (2, 180), 'labels': (2, 180)}


## 5. Evaluation: the same 200 questions for every model

I evaluated the original Qwen and each fine-tuned adapter on the **first 200 questions** from the official GSM8K test split.

For consistency with my original experiments, the evaluation prompt asks for a boxed final answer, although the training prompt uses `Final answer:`. The extractor recognizes both formats. This prompt difference is a limitation of the experiment, so I have kept it visible rather than silently changing the evaluation.

The model generates its answer without sampling (`do_sample=False`). I extract the last recognizable numerical answer and compare it with the expected number.

In [10]:
def extract_final_answer(text):
    patterns = [
        r"\\boxed\{\s*\$?\s*(-?[\d,]+(?:\.\d+)?)\s*\}",
        r"Final answer:\s*\$?\s*(-?[\d,]+(?:\.\d+)?)",
        r"The answer is\s*\$?\s*(-?[\d,]+(?:\.\d+)?)",
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        if matches:
            return float(matches[-1].replace(",", ""))
    return None

def generate_math_solution(question, model):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a mathematics tutor. "
                "Solve the problem step by step. "
                "End with the final numerical answer in \\boxed{}."
            ),
        },
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(model.device)

    model.eval()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

def evaluate_qwen(eval_dataset, model, model_name):
    rows = []
    for index, example in enumerate(tqdm(eval_dataset, desc=model_name)):
        expected = extract_final_answer(example["answer"])
        solution = generate_math_solution(example["question"], model)
        predicted = extract_final_answer(solution)
        correct = (
            expected is not None
            and predicted is not None
            and abs(predicted - expected) < 1e-6
        )
        rows.append({
            "question_number": index + 1,
            "question": example["question"],
            "expected": expected,
            "predicted": predicted,
            "correct": correct,
            "solution": solution,
        })

    results = pd.DataFrame(rows)
    print(f"{model_name}: {results['correct'].sum()}/{len(results)} "
          f"({results['correct'].mean():.2%})")
    return results

eval_dataset = test_clean.select(range(200))
print("Test questions used:", len(eval_dataset))

Test questions used: 200


## 6. V1 — LoRA on attention query and value projections

My first experiment used LoRA on just `q_proj` and `v_proj`. This is a small adapter: only about **1.09 million parameters** were trainable.

I trained for two epochs with an effective batch size of eight (batch size 1 × gradient accumulation 8).

In [11]:
model = load_fresh_model()

v1_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, v1_lora)
model.print_trainable_parameters()

v1_args = TrainingArguments(
    output_dir="./checkpoints/qwen_math_v1",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

v1_trainer = Trainer(
    model=model,
    args=v1_args,
    train_dataset=train_tokenized,
    eval_dataset=validation_tokenized,
    data_collator=math_data_collator,
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


### Train V1

This cell runs the full two-epoch experiment. On my laptop, the original V1 training took roughly 99 minutes. You can skip it if you only want to read the notebook.

In [ ]:
v1_train_result = v1_trainer.train()
v1_trainer.save_model("./adapters/qwen_math_v1")
tokenizer.save_pretrained("./adapters/qwen_math_v1")

('./adapters/qwen_math_v1/tokenizer_config.json',
 './adapters/qwen_math_v1/chat_template.jinja',
 './adapters/qwen_math_v1/tokenizer.json')

### Evaluate V1 against the original model

`disable_adapter()` temporarily turns off LoRA, so both models can be tested with the same evaluation function. We save the individual predictions as CSV files for later error analysis.

In [ ]:
model = v1_trainer.model
model.eval()

v1_results = evaluate_qwen(eval_dataset, model, "QLoRA V1")

with model.disable_adapter():
    base_results = evaluate_qwen(eval_dataset, model, "Original Qwen")

v1_results.to_csv("v1_results_200.csv", index=False)
base_results.to_csv("base_results_200.csv", index=False)

### V1 result from the original run

| Model | Correct / 200 | Accuracy |
|---|---:|---:|
| Original Qwen | 163 | 81.5% |
| QLoRA V1 | 163 | 81.5% |

V1 learned from the training data, but it did **not** improve accuracy over the original model on this test sample. Its final validation loss was approximately **0.5417**.

Before moving to V4, the notebook clears the V1 model from GPU memory. This is important on an 8 GB card.

In [ ]:
del v1_trainer, model
gc.collect()
torch.cuda.empty_cache()
print("V1 model cleared from GPU memory.")

## 7. V4 — LoRA on seven projection modules

For V4, I kept the same rank and alpha but expanded LoRA to seven modules: four attention projections and three feed-forward projections.

That increased the trainable parameter count to about **9.23 million**. I wanted to find out whether giving the adapter more room to learn would improve mathematical accuracy.

In [ ]:
model = load_fresh_model()

v4_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, v4_lora)
model.print_trainable_parameters()

v4_args = TrainingArguments(
    output_dir="./checkpoints/qwen_math_v4",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

v4_trainer = Trainer(
    model=model,
    args=v4_args,
    train_dataset=train_tokenized,
    eval_dataset=validation_tokenized,
    data_collator=math_data_collator,
)

### Train and evaluate V4

V4 uses the same prepared dataset and evaluation function as V1. The main change is which modules receive LoRA adapters. The batch configuration also differs, so this is a practical comparison rather than a perfectly controlled single-variable experiment.

In [ ]:
v4_train_result = v4_trainer.train()
v4_trainer.save_model("./adapters/qwen_math_v4")
tokenizer.save_pretrained("./adapters/qwen_math_v4")

model = v4_trainer.model
model.eval()
v4_results = evaluate_qwen(eval_dataset, model, "QLoRA V4")
v4_results.to_csv("v4_results_200.csv", index=False)

## 8. Compare the models

The results below are from the completed experiments, not numbers generated by executing this clean notebook.

| Model | Trainable parameters | Final validation loss | Correct / 200 | Accuracy |
|---|---:|---:|---:|---:|
| Original Qwen | — | — | 163 | **81.5%** |
| V1 (2 modules) | 1,089,536 | 0.5417 | 163 | **81.5%** |
| V4 (7 modules) | 9,232,384 | **0.4935** | 138 | **69.0%** |

V4's validation loss was lower, but its test accuracy fell by **12.5 percentage points** compared with the original model.

This taught me an important lesson: predicting the reference solution more accurately during training is not the same as solving unseen word problems more accurately.

### Which questions did V4 fix or break?

Comparing each test question individually is more informative than looking only at total accuracy. In the original run:

- **17** questions were wrong for the original model but correct for V4.
- **42** questions were correct for the original model but wrong for V4.
- **121** were correct for both.
- **20** were wrong for both.

The extraction function failed to find an answer in **7** original-model responses and **9** V4 responses. We also inspected V4 mistakes that were genuine reasoning errors, not just formatting problems.

In [ ]:
# Run this after evaluating V4 and the original model.
# If you ran V4 in a new session, load the saved base_results_200.csv first.
if "base_results" not in globals():
    base_results = pd.read_csv("base_results_200.csv")

base_correct = base_results["correct"].astype(bool).to_numpy()
v4_correct = v4_results["correct"].astype(bool).to_numpy()

print("V4 fixed:", int((~base_correct & v4_correct).sum()))
print("V4 regressed:", int((base_correct & ~v4_correct).sum()))
print("Both correct:", int((base_correct & v4_correct).sum()))
print("Both wrong:", int((~base_correct & ~v4_correct).sum()))
print("Base extraction failures:", int(base_results["predicted"].isna().sum()))
print("V4 extraction failures:", int(v4_results["predicted"].isna().sum()))

## 9. What I learned

The original Qwen model was already strong on this small GSM8K test sample. My V1 adapter matched it, while V4 performed worse despite achieving the best validation loss.

I wouldn't claim that expanding LoRA modules always hurts mathematics models. These are results from **one base model, one dataset, particular training settings, and 200 test questions**. The experiments also used different batch settings, and the evaluation prompt didn't exactly match the training prompt.

Still, the project gave me hands-on experience with quantization, LoRA, prompt masking, dynamic padding, model evaluation and error analysis. More importantly, it reminded me to measure the outcome I actually care about—not just the training loss.

**Status:** Experiments finished. I have not included the later GRPO exploration or the intermediate V2/V3 runs in this notebook.